# nb33 - Squeeze without retraining: energy-dependent calibration + mega-ensemble

nb32 record: W=4 3-seed ensemble 0.0452, per-bin 0.068/0.051/0.038/0.038/0.038/0.038. Two free levers remain, both inference-only from the saved nb32 checkpoints: (1) every result so far uses ONE global linear log-calibration over the whole spectrum, so an energy-dependent residual bias inflates every bin - replace it with a monotone (isotonic) calibration fit on the validation split, binned by PREDICTED energy (no truth leakage); (2) combine all six models (W=3 and W=4, seeds 0-2) into one ensemble - different windows make partly-decorrelated errors.

Anchors: nb32 W4-ens 0.0452 | per-bin targets 0.06/0.045/0.035/0.032/0.030/0.030.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd, uproot, awkward as ak
import torch, torch.nn as nn
from sklearn.isotonic import IsotonicRegression
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
MB = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'
DEVICE = os.environ.get('NB33_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB33_MODE', 'full')
if MODE == 'smoke': MB = MB[:8]
THRESH = 2.49
print('device', DEVICE, '| mode', MODE, '|', len(MB), 'files | ckpts:', sorted(p.name for p in CKPT.glob('nb32_*.pt')))

device cuda | mode full | 94 files | ckpts: ['nb32_W3_s0.pt', 'nb32_W3_s1.pt', 'nb32_W3_s2.pt', 'nb32_W4_s0.pt', 'nb32_W4_s1.pt', 'nb32_W4_s2.pt']


In [2]:
TK = ['cell_x','cell_y','energy','cell_energies_front','cell_energies_back',
      'cell_times_front','cell_times_back','imodx','jmody']
AUX = ['sig_flux_prod_vertex_z','sig_flux_eTot']
def event_geom(cc):
    x, yy, e = cc['cell_x'], cc['cell_y'], cc['energy']
    ix, iy = cc['imodx'], cc['jmody']
    seed = int(np.argmax(e))
    pts = np.stack([x, yy], 1)
    pitch = np.full(len(x), np.nan)
    for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
        sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
        if len(p) >= 2:
            d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
            pitch[sel] = np.median(np.min(d, axis=1))
    fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
    pitch[~np.isfinite(pitch)] = fill
    ps = pitch[seed]
    ei = (x - x[seed]) / ps; ej = (yy - yy[seed]) / ps
    di = np.round(ei).astype(int); dj = np.round(ej).astype(int)
    ok = (np.abs(ei - di) < 0.15) & (np.abs(ej - dj) < 0.15)
    return seed, ps, di, dj, ok
def build_grid(files, label):
    EV = []
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TK + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        et_all = ak.to_numpy(a['sig_flux_eTot']).astype(float)
        for i in np.flatnonzero((vz < 100.0) & (et_all >= 1.0) & (et_all <= 100.0)):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TK}
            e = cc['energy']
            if len(e) < 3: continue
            seed, ps, di, dj, ok = event_geom(cc)
            if ok.mean() < 0.5 or not ok[seed]: continue
            tf = cc['cell_times_front']; tb = cc['cell_times_back']
            tf = np.where(np.isfinite(tf) & (tf != 0) & (np.abs(tf) < 1e4), tf, np.nan)
            tb = np.where(np.isfinite(tb) & (tb != 0) & (np.abs(tb) < 1e4), tb, np.nan)
            EV.append(dict(di=di[ok].astype(np.int16), dj=dj[ok].astype(np.int16),
                           e=e[ok].astype(np.float32),
                           fr=cc['cell_energies_front'][ok].astype(np.float32),
                           bk=cc['cell_energies_back'][ok].astype(np.float32),
                           tf=tf[ok].astype(np.float32), tb=tb[ok].astype(np.float32),
                           ps=float(ps), reg=int(np.argmin(np.abs(PITCH - ps))),
                           Etrue=float(et_all[i])))
    print(f'{label}: {len(EV)} events')
    return EV
t0 = time.time()
ME = build_grid(MB, 'minbias')
print(f'build {time.time()-t0:.0f}s')

minbias: 72554 events
build 94s


In [3]:
def make_windows(W):
    rows = []; keep = []
    for i, ev in enumerate(ME):
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum() < 1: continue
        di, dj, e, fr, bk, tf, tb = (v[m] for v in (ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb']))
        t0f = np.nanmedian(tf) if np.isfinite(tf).any() else 0.0
        t0b = np.nanmedian(tb) if np.isfinite(tb).any() else 0.0
        tfc = np.where(np.isfinite(tf), tf - t0f, 0.0); htf = np.isfinite(tf).astype(np.float32)
        tbc = np.where(np.isfinite(tb), tb - t0b, 0.0); htb = np.isfinite(tb).astype(np.float32)
        rdr = np.hypot(di, dj)
        cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                         np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                         rdr, np.full(len(e), np.log(ev['ps'])), np.clip(tfc, -5, 5), np.clip(tbc, -5, 5)], 1)
        oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, ev['reg']] = 1.0
        tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
        rows.append((tok, float(e.sum()), float(e.max()), ev['Etrue'])); keep.append(i)
    return rows, np.array(keep)
def splits_for(keep):
    remap = -np.ones(len(ME), int); remap[keep] = np.arange(len(keep))
    a, b, t = split(len(ME))
    return (remap[a][remap[a] >= 0], remap[b][remap[b] >= 0], remap[t][remap[t] >= 0])
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96, huber_delta=0.1)
NG = 5; NC = 9
class SubNet(nn.Module):
    def __init__(self, in_dim, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 1))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        fl = self.fhead(h).squeeze(-1)
        w = torch.sigmoid(fl) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
def prep(W):
    rows, keep = make_windows(W)
    ktr, kva, kte = splits_for(keep)
    N = len(rows); L = (2*W+1)**2; IN_DIM = rows[0][0].shape[1]
    y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
    Et = np.array([r[3] for r in rows], np.float32)
    sumE = np.array([r[1] for r in rows], np.float32)
    X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
    G = np.zeros((N, NG), np.float32); Eraw = np.zeros((N, L), np.float32)
    for i, (tok, se, sde, et) in enumerate(rows):
        n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
        e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
        lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
        fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
        G[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat]
    la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
    G = (G - G[ktr].mean(0)) / (G[ktr].std(0) + EPS)
    cont = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
    mean = cont.mean(0); std = cont.std(0) + EPS
    X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
    T = dict(X=torch.from_numpy(X).to(DEVICE), M=torch.from_numpy(M).to(DEVICE),
             G=torch.from_numpy(G).to(DEVICE), Y=torch.from_numpy(y).unsqueeze(1).to(DEVICE),
             E=torch.from_numpy(Eraw).to(DEVICE))
    print(f'W={W}: N {N}, tr/va/te {len(ktr)}/{len(kva)}/{len(kte)}, IN_DIM {IN_DIM}')
    return T, y, Et, ktr, kva, kte, IN_DIM, float(la0), float(lb0)

In [4]:
SEEDS = [0, 1, 2]
PRED = {}
for W in (3, 4):
    T, y, Et, ktr, kva, kte, IN_DIM, la0, lb0 = prep(W)
    for seed in SEEDS:
        ck = CKPT / f'nb32_W{W}_s{seed}.pt'
        if not ck.exists():
            print('missing', ck.name); continue
        model = SubNet(IN_DIM, la0, lb0).to(DEVICE)
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['bstate']); model.eval()
        outs = {}
        with torch.no_grad():
            for name, idx in (('va', kva), ('te', kte)):
                o = []
                for j in range(0, len(idx), 256):
                    b = torch.from_numpy(np.asarray(idx[j:j+256])).to(DEVICE)
                    o.append(model(T['X'][b], T['M'][b], T['G'][b], T['E'][b]).cpu().numpy().ravel())
                outs[name] = np.concatenate(o)
        PRED[(W, seed)] = outs
        print(f'W{W} s{seed}: inferred va {len(outs["va"])} te {len(outs["te"])}')
    PRED[('meta', W)] = (y, Et, kva, kte)
    del T
    if DEVICE == 'cuda': torch.cuda.empty_cache()

W=3: N 72554, tr/va/te 50787/10883/10884, IN_DIM 16


W3 s0: inferred va 10883 te 10884


W3 s1: inferred va 10883 te 10884


W3 s2: inferred va 10883 te 10884


W=4: N 72554, tr/va/te 50787/10883/10884, IN_DIM 16


W4 s0: inferred va 10883 te 10884


W4 s1: inferred va 10883 te 10884


W4 s2: inferred va 10883 te 10884


## Calibrations
`linear` = the nb32 baseline (one line in log space). `isotonic` = monotone map from raw prediction to val truth (binned only by the model's own output, applied unchanged to test). Both fit on validation, evaluated on test.

In [5]:
def eval_all(W, raws_va, raws_te):
    y, Et, kva, kte = PRED[('meta', W)]
    res = {}
    a, b = np.polyfit(raws_va, y[kva], 1)
    res['linear'] = np.exp(a * raws_te + b)
    iso = IsotonicRegression(out_of_bounds='clip').fit(raws_va, y[kva])
    res['isotonic'] = np.exp(iso.predict(raws_te))
    return {k: resolution(v, Et[kte])['sigma_eff'] for k, v in res.items()}, res
rows = []
ENS_TE = {}
for W in (3, 4):
    va_stack = [PRED[(W, s)]['va'] for s in SEEDS if (W, s) in PRED]
    te_stack = [PRED[(W, s)]['te'] for s in SEEDS if (W, s) in PRED]
    for s, (v, t) in enumerate(zip(va_stack, te_stack)):
        sig, _ = eval_all(W, v, t)
        rows.append(dict(model=f'W{W}_s{s}', **sig))
    sig, preds = eval_all(W, np.mean(va_stack, 0), np.mean(te_stack, 0))
    rows.append(dict(model=f'W{W}_ens{len(va_stack)}', **sig))
    ENS_TE[W] = (np.mean(va_stack, 0), np.mean(te_stack, 0), preds)
R = pd.DataFrame(rows); print(R.to_string(index=False))

  model  linear  isotonic
  W3_s0  0.0472    0.0504
  W3_s1  0.0489    0.0514
  W3_s2  0.0479    0.0506
W3_ens3  0.0462    0.0486
  W4_s0  0.0471    0.0497
  W4_s1  0.0459    0.0478
  W4_s2  0.0465    0.0482
W4_ens3  0.0450    0.0471


## Cross-window mega-ensemble
W=3 and W=4 test splits index the same events (same base split remapped per window); align on the intersection by original event id and average calibrated energies.

In [6]:
def keep_ids(W):
    rows_, keep = make_windows(W)
    return keep
k3 = keep_ids(3); k4 = keep_ids(4)
y3, Et3, kva3, kte3 = PRED[('meta', 3)]
y4, Et4, kva4, kte4 = PRED[('meta', 4)]
orig3 = k3[kte3]; orig4 = k4[kte4]
common, i3, i4 = np.intersect1d(orig3, orig4, return_indices=True)
print(f'common test events: {len(common)} (W3 {len(orig3)}, W4 {len(orig4)})')
p3 = ENS_TE[3][2]['isotonic'][i3]; p4 = ENS_TE[4][2]['isotonic'][i4]
Etc = Et3[kte3][i3]
assert np.allclose(Etc, Et4[kte4][i4])
for name, pe in (('W3 iso', p3), ('W4 iso', p4), ('mega (W3+W4)/2', 0.5 * (p3 + p4)),
                 ('mega geo', np.sqrt(p3 * p4))):
    print(f'{name:16s} sigma_eff {resolution(pe, Etc)["sigma_eff"]:.4f}')

common test events: 10884 (W3 10884, W4 10884)
W3 iso           sigma_eff 0.0486
W4 iso           sigma_eff 0.0471
mega (W3+W4)/2   sigma_eff 0.0460
mega geo         sigma_eff 0.0461


## Final per-bin table (best combination)

In [7]:
cands = {'W4_ens_iso': (ENS_TE[4][2]['isotonic'], Et4[kte4]),
         'W4_ens_linear': (ENS_TE[4][2]['linear'], Et4[kte4]),
         'mega_iso': (0.5 * (p3 + p4), Etc)}
best_name = min(cands, key=lambda k: resolution(*cands[k])['sigma_eff'])
pe, te_e = cands[best_name]
print(f'best: {best_name} overall {resolution(pe, te_e)["sigma_eff"]:.4f}')
print('targets:            0.06 / 0.045 / 0.035 / 0.032 / 0.030 / 0.030')
edges = np.quantile(te_e, np.linspace(0, 1, 7))
for i in range(6):
    hi = edges[i+1] + (1e-9 if i == 5 else 0)
    mm = (te_e >= edges[i]) & (te_e < hi)
    print(f'  E {edges[i]:6.1f}-{edges[i+1]:6.1f} GeV: {resolution(pe[mm], te_e[mm])["sigma_eff"]:.4f}  (n={int(mm.sum())})')
np.save(OUT / 'nb33_best_pred.npy', pe); np.save(OUT / 'nb33_best_true.npy', te_e)
pd.DataFrame([dict(best=best_name, sigma_eff=resolution(pe, te_e)['sigma_eff'])]).to_csv(OUT / 'nb33_final.csv', index=False)

best: W4_ens_linear overall 0.0450
targets:            0.06 / 0.045 / 0.035 / 0.032 / 0.030 / 0.030
  E    2.2-  10.7 GeV: 0.0684  (n=1814)
  E   10.7-  17.4 GeV: 0.0513  (n=1814)
  E   17.4-  24.0 GeV: 0.0376  (n=1814)
  E   24.0-  34.1 GeV: 0.0383  (n=1814)
  E   34.1-  53.1 GeV: 0.0376  (n=1814)
  E   53.1- 100.0 GeV: 0.0385  (n=1814)
